In [25]:
!pip install fvcore -q
from fvcore.nn import FlopCountAnalysis

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

In [27]:

def get_dataloaders(dataset_name, batch_size):
    # Keep native 28x28 size to match your reference FLOPs
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == "MNIST":
        full_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else:
        full_ds = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    train_loader = torch.utils.data.DataLoader(full_ds, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

In [30]:
def get_model_metrics(m_name, train_loader, test_loader, optimizer_type, lr, device_name, epochs=5):
    device = torch.device(device_name)

    # Model Setup modified for 28x28 grayscale
    if m_name == "ResNet-18":
        model = torchvision.models.resnet18(num_classes=10)
    else:
        model = torchvision.models.resnet50(num_classes=10)

    # Modify for 1-channel 28x28 input
    model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = torch.nn.Identity()
    model.to(device)

    # 1. FLOPs Calculation (28x28 input)
    inputs = torch.randn(1, 1, 28, 28).to(device)
    flops_counter = FlopCountAnalysis(model, inputs)
    total_flops = flops_counter.total() / 1e9  # GFLOPs

    # 2. Optimizer & Criterion
    optimizer = torch.optim.SGD(model.parameters(), lr=lr) if optimizer_type == 'SGD' else torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    # 3. Training Loop
    start_time = time.time()
    model.train()
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    if device.type == 'cuda':
        torch.cuda.synchronize()
    train_time_ms = (time.time() - start_time) * 1000

    # 4. Accuracy
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = 100. * correct / len(test_loader.dataset)
    return accuracy, train_time_ms, total_flops

In [31]:
train_loader, test_loader = get_dataloaders("FashionMNIST", batch_size=16)

# Execution Loop
hardware_options = ['cuda', 'cpu']
models_to_test = ["ResNet-18", "ResNet-50"]
optimizers = ["SGD", "Adam"]
q2_results = []

for hw in hardware_options:
    for m_name in models_to_test:
        for opt in optimizers:
            print(f"Testing {m_name} | {hw.upper()} | {opt}...")
            acc, t_ms, flops = get_model_metrics(
                m_name, train_loader, test_loader, opt, 0.001, hw, epochs=5
            )

            q2_results.append({
                "Compute": hw.upper(),
                "Batch Size": 16,
                "Optimizer": opt,
                "Learning Rate": 0.001,
                "Model": m_name,
                "Test Accuracy (%)": round(acc, 2),
                "Train Time (ms)": int(t_ms),
                "FLOPs (GFLOPs)": round(flops, 4)
            })

# Final Table Output
df_q2 = pd.DataFrame(q2_results)
df_q2.to_csv("q2_hardware_results.csv", index=False)
print(df_q2)

Testing ResNet-18 | CUDA | SGD...


Testing ResNet-18 | CUDA | Adam...


Testing ResNet-50 | CUDA | SGD...


Testing ResNet-50 | CUDA | Adam...


Testing ResNet-18 | CPU | SGD...


KeyboardInterrupt: 